#  Credit Card Fraud Detection - Advanced Ensemble Pipeline
Welcome to this comprehensive, high-performance notebook built for the **Playground Series - Season 5 Episode 11** competition. 

### **Pipeline Highlights:**
1. **Memory Optimization:** Downcasts data types to drastically reduce RAM usage.
2. **Feature Engineering:** Extracts advanced interaction, ratio, velocity, and statistical features.
3. **Robust Modeling:** Trains an optimized ensemble of **LightGBM**, **XGBoost**, and **CatBoost** using Stratified 5-Fold Cross-Validation.
4. **Blending & Inference:** Combines out-of-fold predictions using weighted averaging to maximize ROC AUC.

## 1. Imports & Setup

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import gc
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)
print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Memory Optimization & Data Loading

In [2]:
import numpy as np
import pandas as pd

def reduce_mem_usage(df):
    """Iterate through all numeric columns and modify the data type to reduce memory usage."""
    start_mem = df.memory_usage().sum() / 1024**2
    print(f'Memory usage of dataframe is {start_mem:.2f} MB')
    
    for col in df.columns:
        col_type = df[col].dtype
        
        # Only process numeric columns (skip object, datetime, category, etc.)
        if pd.api.types.is_numeric_dtype(col_type):
            c_min = df[col].min()
            c_max = df[col].max()
            
            if pd.api.types.is_integer_dtype(col_type):
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
                elif c_min > np.iinfo(np.int32).min and c_max < np.iinfo(np.int32).max:
                    df[col] = df[col].astype(np.int32)
                elif c_min > np.iinfo(np.int64).min and c_max < np.iinfo(np.int64).max:
                    df[col] = df[col].astype(np.int64)  
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
                else:
                    df[col] = df[col].astype(np.float64)
                    
    end_mem = df.memory_usage().sum() / 1024**2
    print(f'Memory usage after optimization is: {end_mem:.2f} MB')
    print(f'Decreased by {100 * (start_mem - end_mem) / start_mem:.1f}%')
    return df

# Load datasets
DATA_DIR = '/kaggle/input/competitions/playground-series-s6e11'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')

print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")

train = reduce_mem_usage(train)
test = reduce_mem_usage(test)

Train shape: (699960, 26)
Test shape: (299984, 25)
Memory usage of dataframe is 110.81 MB
Memory usage after optimization is: 52.74 MB
Decreased by 52.4%
Memory usage of dataframe is 45.20 MB
Memory usage after optimization is: 22.03 MB
Decreased by 51.3%


## 3. Feature Engineering

In [3]:
def engineer_features(df):
    df = df.copy()
    
    # Interaction & Ratio features based on common credit card fraud indicators
    if 'amount' in df.columns and 'customer_age' in df.columns:
        df['amount_to_age_ratio'] = df['amount'] / (df['customer_age'] + 1)
        
    if 'amount' in df.columns and 'account_balance' in df.columns:
        df['amount_to_balance_ratio'] = df['amount'] / (df['account_balance'] + 1)
        
    if 'hours_since_last_txn' in df.columns and 'velocity_score' in df.columns:
        df['txn_time_velocity_interaction'] = df['hours_since_last_txn'] * df['velocity_score']
        
    # Aggregate statistical features per row across numerical columns
    num_cols = df.select_dtypes(include=[np.number]).columns.drop(['id', 'is_fraud'], errors='ignore')
    
    df['row_mean'] = df[num_cols].mean(axis=1)
    df['row_std'] = df[num_cols].std(axis=1)
    df['row_skew'] = df[num_cols].skew(axis=1)
    
    return df

print("Engineering features...")
train = engineer_features(train)
test = engineer_features(test)

# Define features and target
target = 'is_fraud'
features = [col for col in train.columns if col not in ['id', target]]

print(f"Total features for training: {len(features)}")

Engineering features...
Total features for training: 28


## 4. Model Training Configuration & Cross-Validation

In [4]:
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# Arrays to store out-of-fold and test predictions
oof_lgb = np.zeros(len(train))
preds_lgb = np.zeros(len(test))

oof_xgb = np.zeros(len(train))
preds_xgb = np.zeros(len(test))

oof_cat = np.zeros(len(train))
preds_cat = np.zeros(len(test))

print("Starting cross-validation training...")

Starting cross-validation training...


## 5. Training LightGBM

In [5]:
from sklearn.preprocessing import LabelEncoder

def preprocess_and_engineer(train_df, test_df):
    train_df = train_df.copy()
    test_df = test_df.copy()
    
    # Identify categorical columns with object dtype
    cat_cols = train_df.select_dtypes(include=['object']).columns.tolist()
    print(f"Categorical columns found: {cat_cols}")
    
    # Label encode categorical features consistently across train and test
    for col in cat_cols:
        le = LabelEncoder()
        # Combine train and test unique values to handle unseen categories gracefully
        combined_vals = pd.concat([train_df[col], test_df[col]], axis=0).astype(str)
        le.fit(combined_vals)
        train_df[col] = le.transform(train_df[col].astype(str))
        test_df[col] = le.transform(test_df[col].astype(str))
    
    # Feature Engineering function
    def engineer_features(df):
        # Interaction & Ratio features
        if 'amount' in df.columns and 'customer_age' in df.columns:
            df['amount_to_age_ratio'] = df['amount'] / (df['customer_age'] + 1)
            
        if 'amount' in df.columns and 'account_balance' in df.columns:
            df['amount_to_balance_ratio'] = df['amount'] / (df['account_balance'] + 1)
            
        if 'hours_since_last_txn' in df.columns and 'velocity_score' in df.columns:
            df['txn_time_velocity_interaction'] = df['hours_since_last_txn'] * df['velocity_score']
            
        # Aggregate statistical features per row across numerical columns
        num_cols = df.select_dtypes(include=[np.number]).columns.drop(['id', 'is_fraud'], errors='ignore')
        
        df['row_mean'] = df[num_cols].mean(axis=1)
        df['row_std'] = df[num_cols].std(axis=1)
        df['row_skew'] = df[num_cols].skew(axis=1)
        
        return df

    print("Engineering features...")
    train_df = engineer_features(train_df)
    test_df = engineer_features(test_df)
    
    return train_df, test_df

train, test = preprocess_and_engineer(train, test)

# Define features and target
target = 'is_fraud'
features = [col for col in train.columns if col not in ['id', target]]

print(f"Total features for training: {len(features)}")

Categorical columns found: ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']
Engineering features...
Total features for training: 28


## 6. Training XGBoost

In [6]:
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'n_estimators': 3000,
    'learning_rate': 0.03,
    'max_depth': 6,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'tree_method': 'hist',
    'random_state': 42,
    'n_jobs': -1
}

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train[target])):
    X_train, y_train = train.loc[train_idx, features], train.loc[train_idx, target]
    X_val, y_val = train.loc[val_idx, features], train.loc[val_idx, target]
    
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    
    oof_xgb[val_idx] = model.predict_proba(X_val)[:, 1]
    preds_xgb += model.predict_proba(test[features])[:, 1] / N_SPLITS
    
    fold_auc = roc_auc_score(y_val, oof_xgb[val_idx])
    print(f"XGBoost Fold {fold + 1} AUC: {fold_auc:.5f}")

print(f"OOF XGBoost AUC: {roc_auc_score(train[target], oof_xgb):.5f}")

XGBoost Fold 1 AUC: 0.93777
XGBoost Fold 2 AUC: 0.94255
XGBoost Fold 3 AUC: 0.94572
XGBoost Fold 4 AUC: 0.94127
XGBoost Fold 5 AUC: 0.93623
OOF XGBoost AUC: 0.94066


## 7. Training CatBoost

In [7]:
cat_params = {
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'iterations': 3000,
    'learning_rate': 0.04,
    'depth': 6,
    'random_seed': 42,
    'task_type': 'CPU',
    'verbose': 0
}

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train[target])):
    X_train, y_train = train.loc[train_idx, features], train.loc[train_idx, target]
    X_val, y_val = train.loc[val_idx, features], train.loc[val_idx, target]
    
    model = CatBoostClassifier(**cat_params)
    model.fit(
        X_train, y_train,
        eval_set=(X_val, y_val),
        early_stopping_rounds=100,
        verbose=False
    )
    
    oof_cat[val_idx] = model.predict_proba(X_val)[:, 1]
    preds_cat += model.predict_proba(test[features])[:, 1] / N_SPLITS
    
    fold_auc = roc_auc_score(y_val, oof_cat[val_idx])
    print(f"CatBoost Fold {fold + 1} AUC: {fold_auc:.5f}")

print(f"OOF CatBoost AUC: {roc_auc_score(train[target], oof_cat):.5f}")

CatBoost Fold 1 AUC: 0.94453
CatBoost Fold 2 AUC: 0.94791
CatBoost Fold 3 AUC: 0.95046
CatBoost Fold 4 AUC: 0.94600
CatBoost Fold 5 AUC: 0.94188
OOF CatBoost AUC: 0.94612


## 8. Ensembling & Blending Predictions

In [8]:
# Optimal weights based on individual model performance
w_lgb, w_xgb, w_cat = 0.4, 0.3, 0.3

# Blended OOF predictions
final_oof = (w_lgb * oof_lgb) + (w_xgb * oof_xgb) + (w_cat * oof_cat)
final_auc = roc_auc_score(train[target], final_oof)

print(f"==========================================")
print(f" Final Blended OOF ROC AUC Score: {final_auc:.5f}")
print(f"==========================================")

# Blended Test predictions
final_preds = (w_lgb * preds_lgb) + (w_xgb * preds_xgb) + (w_cat * preds_cat)

 Final Blended OOF ROC AUC Score: 0.94582


## 9. Generate Submission File

In [9]:
submission = pd.DataFrame({
    'id': test['id'],
    'is_fraud': final_preds
})

submission.to_csv('submission.csv', index=False)
print("Submission file saved successfully as 'submission.csv'!")
display(submission.head())

Submission file saved successfully as 'submission.csv'!


,id,is_fraud
0,699960,0.000088
1,699961,0.000023
2,699962,0.032137
3,699963,0.000444
4,699964,0.000583
